Class 5 - Windows Function of SQL 

In [0]:
%sql
-- Orders table backup 
CREATE TABLE orders_backup
USING DELTA 
LOCATION 'dbfs:/FileStore/backup/orders';

In [0]:
%sql
-- Customers table backup 
CREATE TABLE customers_backup
USING DELTA 
LOCATION 'dbfs:/FileStore/backup/customers';

In [0]:
%sql
-- order_details table backup 
CREATE TABLE order_details_backup
USING DELTA 
LOCATION 'dbfs:/FileStore/backup/order_details';

In [0]:
%sql 
-- Gives each row a unique number based on order date 
SELECT 
order_id,
order_date, 
ROW_NUMBER() OVER 
(ORDER BY order_date) AS row_num
FROM orders_backup;

In [0]:
%sql 
-- Real Life Use Case: Show first product bought by each customer

SELECT c.first_name,
od.product_name, o.order_id, 
ROW_NUMBER() OVER (PARTITION BY c.customer_id ORDER BY o.order_date) AS PURCHASE_SEQUENCE
FROM customers_backup c 
JOIN orders_backup o ON c.customer_id = o.customer_id
JOIN order_details_backup od ON o.order_id = od.order_id

-- The First Order is that where the purchase_sequence = 1


In [0]:
%sql 
-- RANK() 
-- Rank's rows by quantity, ties get the same rank, but gaps are left in the sequence.
SELECT 
  product_name, quantity, 
  RANK() OVER 
  (ORDER BY quantity DESC) as RANKING 
FROM order_details_backup;

In [0]:
%sql 
-- Real Life UseCase: assign bonus based on best selling products in each order  
SELECT order_id, product_name, quantity,
RANK() OVER 
(PARTITION BY order_id ORDER BY quantity DESC) RANKING
FROM order_details_backup;